# Μάθημα 13 - Μνήμη Πράκτορα με Γραφήματα Γνώσης Cognee


## Ρύθμιση

Αυτό το σημειωματάριο δείχνει πώς να δημιουργήσετε έναν έξυπνο **βοηθό κωδικοποίησης** με μνήμη που διατηρείται χρησιμοποιώντας τα γραφήματα γνώσης [**Cognee**](https://www.cognee.ai/) και το **Microsoft Agent Framework** (MAF).

Το Cognee μετατρέπει μη δομημένο κείμενο σε ένα δομημένο γράφημα γνώσης που μπορεί να ερωτηθεί, υποστηριζόμενο από ενσωματώσεις διανυσμάτων — δίνοντας στον πράκτορά σας μια πλούσια, με επίγνωση σχέσεων μακροχρόνια μνήμη.

### Τι θα μάθετε
1. **Κατασκευή Γραφημάτων Γνώσης**: Μετατρέψτε προφίλ προγραμματιστών και βέλτιστες πρακτικές σε δομημένη, ερωτήσιμη γνώση.
2. **Ενσωμάτωση Cognee με MAF**: Χρησιμοποιήστε τις συναρτήσεις `@tool` για να επιτρέψετε σε έναν πράκτορα MAF να ερωτά το γράφημα γνώσης του Cognee.
3. **Συνομιλίες με Ενημέρωση Συνεδρίας**: Διατηρήστε το πλαίσιο μέσα από πολλαπλές ερωτήσεις στην ίδια συνεδρία.
4. **Μακροχρόνια Μνήμη**: Διατηρήστε σημαντική γνώση μεταξύ των συνεδριών και ανακαλέστε την σε νέες συνομιλίες.

### Προαπαιτούμενα
- Python 3.9+
- Redis σε τοπικό περιβάλλον (`docker run -d -p 6379:6379 redis`) για διαχείριση συνεδριών
- Κλειδί API για LLM (π.χ. OpenAI) — ρυθμίστε το `LLM_API_KEY` στο `.env`
- `CACHING=true` στο `.env` (απαιτείται για τις συνεδρίες Cognee)
- Ένα έργο Microsoft Foundry με αναπτυγμένο μοντέλο συνομιλίας
- `AZURE_AI_PROJECT_ENDPOINT` και `AZURE_AI_MODEL_DEPLOYMENT_NAME` στο `.env`
- Επαλήθευση σύνδεσης μέσω Azure CLI (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")


In [ ]:
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ FoundryChatClient created")


## Τύποι Μνήμης Πράκτορα

Αυτό το σημειωματάριο εξερευνά τους ίδιους τρεις τύπους μνήμης από το κύριο σημειωματάριο Μαθήματος 13, αλλά χρησιμοποιεί το Cognee ως το σύστημα μακροπρόθεσμης μνήμης:

| Τύπος Μνήμης | Μηχανισμός | Διάρκεια Ζωής |
|---|---|---|
| **Εργασίας** | `agent.create_session()` (MAF) | Μία μόνο συνομιλία |
| **Βραχυπρόθεσμη** | Κρυφή μνήμη συνεδρίας Cognee (Redis) | Μία μόνο συνεδρία |
| **Μακροπρόθεσμη** | Γραφικό γνώσης Cognee + διανύσματα | Μόνιμη |

### Αρχιτεκτονική Μνήμης του Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Ετοιμάστε το Cognee Storage


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Μέρος 1 — Δημιουργία της Βάσης Γνώσης

Εισάγουμε τρεις τύπους δεδομένων για να δημιουργήσουμε μια ολοκληρωμένη βάση γνώσεων για τον βοηθό προγραμματισμού μας:

1. **Προφίλ Προγραμματιστή** — προσωπική εξειδίκευση και τεχνικό υπόβαθρο  
2. **Καλύτερες Πρακτικές Python** — το Ζεν της Python με πρακτικές οδηγίες  
3. **Ιστορικές Συνομιλίες** — παλαιότερες συνεδρίες ερωτήσεων και απαντήσεων μεταξύ προγραμματιστών και βοηθών AI


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Οπτικοποίηση του Γράφου Γνώσης

Το Cognee μπορεί να αποδώσει μια διαδραστική HTML οπτικοποίηση των οντοτήτων και των σχέσεων που εξήγαγε.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Εμπλουτίστε τη Μνήμη με τη Memify

Η `memify()` αναλύει το γράφο γνώσης και δημιουργεί ευφυείς κανόνες — εντοπίζοντας μοτίβα, βέλτιστες πρακτικές και σχέσεις μεταξύ εννοιών.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Μέρος 2 — Πράκτορας MAF με τα Εργαλεία Cognee

Τώρα δημιουργούμε έναν πράκτορα MAF που μπορεί να ερωτά το γράφημα γνώσης του Cognee μέσω των λειτουργιών `@tool`. Αυτό επιτρέπει στον πράκτορα να αξιοποιήσει όλη τη δύναμη της γραφο-ενημερωμένης σημασιολογικής αναζήτησης ενώ διατηρεί το πλαίσιο της συνομιλίας μέσω συνεδριών.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = provider.as_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")


## Εργαζόμενη Μνήμη με Συνεδρίες

Η `AgentSession` (που δημιουργείται μέσω `agent.create_session()`) παρέχει εργαζόμενη μνήμη μέσα σε μια συνεδρία. Ο πράκτορας μπορεί να αναφερθεί σε προηγούμενα μηνύματα ενώ ταυτόχρονα κάνει ερωτήματα στο μακροπρόθεσμο γράφο γνώσεων του Cognee.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Νέα Συνεδρία — Η Μακροπρόθεσμη Μνήμη Διατηρείται

Η έναρξη μιας νέας συνεδρίας διαγράφει τη μνήμη εργασίας, αλλά ο γράφος γνώσης Cognee είναι ακόμα διαθέσιμος. Ο πράκτορας μπορεί να ανακτήσει την ίδια μακροπρόθεσμη γνώση σε μια εντελώς νέα συνομιλία.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Περίληψη

Σε αυτό το τετράδιο δημιουργήσατε έναν βοηθό κωδικοποίησης που συνδυάζει τη **μνήμη εργασίας του MAF** (`agent.create_session()`) με το **μακροχρόνιο γράφο γνώσης του Cognee**.

### Τι Μάθατε
1. **Κατασκευή γράφου γνώσης**: Το Cognee εισάγει μη δομημένο κείμενο και κατασκευάζει ένα γράφο + διανυσματική μνήμη.
2. **Εμπλουτισμός γράφου με το memify**: Παράγονται παράγωγα γεγονότα και πλουσιότερες σχέσεις πάνω στον υπάρχοντα γράφο σας.
3. **Ενσωμάτωση MAF + Cognee**: Οι συναρτήσεις `@tool` επιτρέπουν σε έναν πράκτορα MAF να ρωτάει φυσικά τον γράφο του Cognee.
4. **Μνήμη εργασίας + μακροχρόνια μνήμη**: Το `AgentSession` (μέσω `agent.create_session()`) παρέχει πλαίσιο συνεδρίας ενώ το Cognee προσφέρει μόνιμη γνώση.
5. **Φιλτραρισμένη αναζήτηση με NodeSets**: Επικεντρωθείτε σε συγκεκριμένα υποσύνολα του γράφου γνώσης (π.χ. μόνο αρχές).

### Βασικά Συμπεράσματα
- Το **Cognee** μετατρέπει το ακατέργαστο κείμενο σε δομημένη, σχέσεις-ενήμερη μνήμη — ισχυρότερο από ένα απλό διανυσματικό κατάστημα.
- Οι **συναρτήσεις `@tool`** γεφυρώνουν καθαρά τους πράκτορες MAF με εξωτερικά συστήματα γνώσης.
- Το **`AgentSession`** (μέσω `agent.create_session()`) διατηρεί το πλαίσιο ανά συνομιλία ξεχωριστά από τη μακροχρόνια γνώση.
- Ο ίδιος γράφος γνώσης εξυπηρετεί πολλαπλές συνεδρίες και πράκτορες.

### Εφαρμογές στην Πραγματική Ζωή
- **Αναπληρωτές προγραμματιστών**: Ανασκόπηση κώδικα, ανάλυση περιστατικών, βοηθοί αρχιτεκτονικής
- **Αναπληρωτές εξυπηρέτησης πελατών**: Βοηθοί υποστήριξης βασισμένοι σε τεκμηρίωση προϊόντων, Συχνές Ερωτήσεις, και σημειώσεις CRM
- **Εσωτερικοί ειδικοί βοηθοί**: Βοηθοί πολιτικής, νομικοί ή ασφαλείας που σκέφτονται πάνω σε κατευθυντήριες οδηγίες
- **Ενοποιημένα επίπεδα δεδομένων**: Συνδυασμός δομημένων και αδόμητων δεδομένων σε έναν ερωτήσιμο γράφο

### Επόμενα Βήματα
- Πειραματιστείτε με χρονική συνείδηση στο Cognee
- Ορίστε μια οντολογία OWL για την ποιότητα γράφου σε συγκεκριμένους τομείς
- Προσθέστε βρόγχους ανατροφοδότησης χρηστών για βελτίωση της ανάκτησης με το χρόνο
- Κλιμακώστε σε πολυ-πρακτορικά συστήματα που μοιράζονται το ίδιο στρώμα μνήμης Cognee


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
